In [1]:
import getpass
import json
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from anthropic import Anthropic

# Securely prompt for the API key string
api_key = getpass.getpass("Enter your Anthropic API Key: ")
client = Anthropic(api_key=api_key)

print(" Initialization complete. Anthropic client is ready.")

 Initialization complete. Anthropic client is ready.


In [2]:
# 1. DEFINE THE SHARED STATE
class BugReportState(TypedDict):
    ticket_id: str
    description: str
    category: str       # Filled by analyzer: 'frontend' or 'backend'
    ai_reasoning: str   # Filled by analyzer: Claude's explanation
    action_taken: str   # Filled by specialists

# 2. DEFINE THE NODES
def analyzer_node(state: BugReportState) -> dict:
    print(f"\n🔍 [NODE -> ANALYZER]: Ingesting ticket {state['ticket_id']}...")
    
    prompt = f"""Analyze this software bug ticket:
Ticket ID: {state['ticket_id']}
Description: {state['description']}

Classify it strictly into one of two categories: 'frontend' or 'backend'.
Provide a concise, 1-sentence engineering reason explaining your choice.

Return your answer strictly in valid JSON format exactly like this:
{{"category": "frontend/backend", "reason": "Your reason here"}}"""

    # Call Claude for intelligent categorization
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    
    try:
        result = json.loads(response.content[0].text)
        detected_category = result.get("category", "backend").lower().strip()
        reason = result.get("reason", "No explicit reasoning provided.")
    except Exception:
        # Fallback logic if parsing fails
        desc = state["description"].lower()
        detected_category = "frontend" if any(x in desc for x in ["button", "css", "ui", "screen"]) else "backend"
        reason = "Fallback keyword verification engine deployed."
        
    print(f"🤖 [Claude Decision]: Categorized as -> {detected_category.upper()}")
    print(f"💡 [Claude Reason]: {reason}")
    
    return {"category": detected_category, "ai_reasoning": reason}

def frontend_expert_node(state: BugReportState) -> dict:
    print("\n🎨 [NODE -> FRONTEND EXPERT]: Processing UI/UX layout trace...")
    resolution = "Action: Inspected the DOM, updated CSS flexbox constraints, and validated layout rendering consistency."
    return {"action_taken": resolution}

def backend_expert_node(state: BugReportState) -> dict:
    print("\n⚙️ [NODE -> BACKEND EXPERT]: Processing architecture core exception...")
    resolution = "Action: Inspected connection pool thresholds, terminated stale transaction deadlocks, and optimized query indices."
    return {"action_taken": resolution}

In [3]:
# 3. DEFINE THE ROUTER LOGIC
def router_logic(state: BugReportState) -> Literal["route_to_frontend", "route_to_backend"]:
    print(f"\n🔀 [EDGE -> ROUTER]: Evaluating active state category: '{state['category']}'")
    if state["category"] == "frontend":
        print("➡️ Routing path selected: frontend_expert")
        return "route_to_frontend"
    else:
        print("➡️ Routing path selected: backend_expert")
        return "route_to_backend"

# 4. ORCHESTRATE AND COMPILE THE GRAPH
builder = StateGraph(BugReportState)

# Register worker nodes
builder.add_node("analyzer", analyzer_node)
builder.add_node("frontend_expert", frontend_expert_node)
builder.add_node("backend_expert", backend_expert_node)

# Construct edges
builder.add_edge(START, "analyzer")
builder.add_conditional_edges(
    "analyzer",
    router_logic,
    {
        "route_to_frontend": "frontend_expert",
        "route_to_backend": "backend_expert"
    }
)
builder.add_edge("frontend_expert", END)
builder.add_edge("backend_expert", END)

# Compile application binary
bug_triage_app = builder.compile()
print("🎯 Graph workflow compiled successfully and ready for inputs.")

🎯 Graph workflow compiled successfully and ready for inputs.


In [5]:
# Define your test case here
initial_input = {
    "ticket_id": "TICKET-905",
    "description": "Stored procedures are failing intermittently when the database load is high, causing transaction timeouts and data inconsistency issues."
}

print("🚀 Starting the interactive LangGraph pipeline...")
print(f"Initial Ticket: {initial_input['ticket_id']} - {initial_input['description']}\n")

# Use streaming mode to step through the graph nodes sequentially
events = bug_triage_app.stream(initial_input, stream_mode="updates")

for event in events:
    for node_name, state_update in event.items():
        print(f"\n🛑 [PAUSED] Node '{node_name}' has completed execution.")
        print(f"State Delta Update: {json.dumps(state_update, indent=2)}")
        
        # Interactive Gate: Execution pauses right here inside the notebook
        user_input = input("\nPress Enter to ALLOW the pipeline to progress to the next step (or type 'quit' to halt): ")
        if user_input.strip().lower() == 'quit':
            print("❌ Workflow terminated by user request.")
            break

print("\n🏁 Workflow execution trace ended.")

🚀 Starting the interactive LangGraph pipeline...
Initial Ticket: TICKET-905 - Stored procedures are failing intermittently when the database load is high, causing transaction timeouts and data inconsistency issues.


🔍 [NODE -> ANALYZER]: Ingesting ticket TICKET-905...
🤖 [Claude Decision]: Categorized as -> BACKEND
💡 [Claude Reason]: Fallback keyword verification engine deployed.

🔀 [EDGE -> ROUTER]: Evaluating active state category: 'backend'
➡️ Routing path selected: backend_expert

🛑 [PAUSED] Node 'analyzer' has completed execution.
State Delta Update: {
  "category": "backend",
  "ai_reasoning": "Fallback keyword verification engine deployed."
}

⚙️ [NODE -> BACKEND EXPERT]: Processing architecture core exception...

🛑 [PAUSED] Node 'backend_expert' has completed execution.
State Delta Update: {
  "action_taken": "Action: Inspected connection pool thresholds, terminated stale transaction deadlocks, and optimized query indices."
}

🏁 Workflow execution trace ended.
